In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from feature_engineering.order_flow_pipeline import OrderFlowFeatureEngineering
from feature_engineering.order_flow import L2QFeatureEngineering, TradeFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

In [ ]:
# 2. Run pipeline
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=5,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=5,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

pipeline = Pipeline(config)
results = pipeline.run(slice(10))

In [ ]:
# 3. Verify results
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

In [ ]:
# 4. Test Feature Engineering Pipeline
# Create test config for feature engineering
fe_config = PipelineConfig(
    region='us-east-1',
    processing=ProcessingConfig(
        feature_engineering=L2QFeatureEngineering(bar_duration_ms=250)
    ),
    ray=RayConfig(
        runtime_env={"working_dir": src_dir},
        flat_core_count=2,
        memory_per_core_gb=4.0,
        memory_multiplier=2.0,
        cpu_buffer=1
    )
)

# Initialize feature engineering pipeline
fe_pipeline = OrderFlowFeatureEngineering(fe_config)

# Test with a few normalized files
test_files = [(result['output_path'], 0.1) for result in results[:3] if result['message'] == 'success']
print(f"Testing feature engineering with {len(test_files)} files")

# Process files
fe_results = fe_pipeline.process_files(test_files)

# Show results
for result in fe_results:
    if result['message'] == 'success':
        print(f"✓ {result['data_type']}: {result['row_count']:,} features -> {result['output_path']}")
    else:
        print(f"✗ Failed: {result['message'][:100]}")

fe_pipeline.shutdown()